In [5]:
import random
from copy import deepcopy
from datasets import load_dataset, DatasetDict
from pprint import pprint

ORIGINAL_REPO_ID = "apurbapokharel/synthetic_deliberation"
NEW_REPO_ID = "apurbapokharel/distractor_deliberation"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

def create_flat_mcq_split(split):

    data = list(split)  # convert to list of dicts
    n = len(data)

    # diagnosis -> list of example indices
    diag_to_indices = {}
    for i, ex in enumerate(data):
        diag = ex["gold_diagnosis"]
        diag_to_indices.setdefault(diag, []).append(i)

    all_indices = list(range(n))
    new_rows = []

    for idx, ex in enumerate(data):
        gold_diag = ex["gold_diagnosis"]
        correct_reply = ex["reply"]

        # candidates = all rows with different diagnosis than current
        candidate_indices = []
        for diag, idxs in diag_to_indices.items():
            if diag != gold_diag:
                candidate_indices.extend(idxs)

        # if too few, fallback to all except itself
        if len(candidate_indices) < 3:
            candidate_indices = [i for i in all_indices if i != idx]

        # sample 3 distractors (with replacement only if needed)
        if len(candidate_indices) >= 3:
            chosen_indices = random.sample(candidate_indices, 3)
        else:
            chosen_indices = [random.choice(candidate_indices) for _ in range(3)]

        distractors = [data[i]["reply"] for i in chosen_indices]

        # Build a shuffled list of 4 endings
        endings = [correct_reply] + distractors
        random.shuffle(endings)

        # label = index of correct reply
        label = endings.index(correct_reply)

        # Create new row fields
        new_ex = deepcopy(ex)
        new_ex["ending0"] = endings[0]
        new_ex["ending1"] = endings[1]
        new_ex["ending2"] = endings[2]
        new_ex["ending3"] = endings[3]
        new_ex["label"] = label

        # Optional: build a "context" field for easier training later
        # context = f"{ex['problem']}\n\n{ex['history_current']}\n\n{ex['history_neighbour']}"
        # new_ex["context"] = context

        new_rows.append(new_ex)
    return split.from_list(new_rows)


if __name__ == "__main__":
    # 1. Load dataset
    ds = load_dataset(ORIGINAL_REPO_ID)

    # 2. Process each split
    new_splits = {}
    for split_name, split in ds.items():
        print(f"Processing split '{split_name}', {len(split)} examples")
        new_splits[split_name] = create_flat_mcq_split(split)

    new_ds = DatasetDict(new_splits)

    # 3. Push back to HF
    new_ds.push_to_hub(NEW_REPO_ID)

    print(f"\nDataset successfully uploaded to: {NEW_REPO_ID}")


Processing split 'train', 5827 examples
Processing split 'validation', 728 examples
Processing split 'test', 729 examples


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.



Dataset successfully uploaded to: apurbapokharel/distractor_deliberation
